# SE-ResNet-Lite — Attention-augmented, Shrunk ResNet for AoA/AoD

Base paper-er ResNet backbone (64 stacked residual block, 469,393 param) take
modify kore ei jinis-gula test kora:

1. **Fewer blocks** -- 64 theke onek kome anle koto param bache, ar accuracy koto
   drop kore (ba na kore) seta measure kora.
2. **Squeeze-and-Excitation (SE) attention** -- protita residual block-e ekta
   choto channel-attention gate jog kora (khub kom parameter overhead-e), jate
   network nijei shikhte pare kon channel-gulo beshi informative -- ei ashai
   je noisy (low-SNR) input-e eta help korte pare.
3. **Same training/eval protocol as base paper-er ResNet** -- `data_generation()`
   generator (random L, random SNR -15..25, random P/Q/Nt/Nr) diye train, ar
   ঠিক shei `run_inference_and_metrics_resnet`-er moto eval protocol (L=3,
   SNR -10..25, sigma=0.07, M=256) diye test -- jate result-ta sotti tulonajogyo hoy.

**Guruttopurno:** ei model-take original pretrained weight diye load kora
jabe na (architecture change hoyeche), tai eta **scratch theke train** korte
hobe. Ei notebook-e ekta CHOTO smoke-test ache (dekhar jonno shob kisu kaj
kore kina), kintu REAL training (jate real accuracy bojha jay) tomar nijer
Kaggle GPU-te full scale-e চালাতে hobe.

## Part 0 — Setup (self-contained, kono repo path dependency nei)

In [ ]:
import os, sys, time, random as pyrandom
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.layers import (Dense, Conv2D, Input, BatchNormalization, Activation,
                                       Add, Conv2DTranspose, GlobalAveragePooling2D, Reshape, Multiply)
from tensorflow.keras.models import Model

tf.get_logger().setLevel('ERROR')
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

np.random.seed(42)
tf.random.set_seed(42)
pyrandom.seed(42)

print('TensorFlow:', tf.__version__, ' GPU:', tf.config.list_physical_devices('GPU'))

## Part 1 — Physics + data generation (copied verbatim from DL_DOA/src/,
tai kono sys.path/Kaggle-mount somossha hobe na)

In [ ]:
def generate_points(M, delta, max_attempts=10000, rng=None):
    import math
    if rng is None:
        rng = np.random
    points = []
    attempts = 0
    while len(points) < M and attempts < max_attempts:
        x = rng.uniform(0, math.pi); y = rng.uniform(0, math.pi)
        new_point = (x, y)
        too_close = any(math.hypot(new_point[0]-p[0], new_point[1]-p[1]) < delta for p in points)
        if not too_close:
            points.append(new_point)
        attempts += 1
    if len(points) < M:
        raise ValueError(f"Could not generate {M} points with delta={delta} after {max_attempts} attempts.")
    return points


def wrapTo2Pi(x):
    return np.mod(x, 2*np.pi)


def ev(nt, angle):
    k = np.arange(nt)
    vector = (1/np.sqrt(nt)) * np.exp(-1j * np.pi * np.cos(angle) * k)
    return np.expand_dims(vector, axis=-1)


def beamforming_vector_generation_P(P, nt):
    p = np.arange(P)
    cosp = (1/np.pi) * np.angle(np.exp(1j * (2*np.pi/P) * p))
    phi_p = np.arccos(cosp)
    F = np.zeros((nt, P), dtype=complex)
    for idx_p in range(P):
        F[:, idx_p] = np.squeeze(ev(nt, phi_p[idx_p]), -1)
    return F


def beamforming_vector_generation_Q(Q, nr):
    q = np.arange(Q)
    cosq = (1/np.pi) * np.angle(np.exp(-1j * (2*np.pi/Q) * q))
    phi_q = np.arccos(cosq)
    W = np.zeros((nr, Q), dtype=complex)
    for idx_q in range(Q):
        W[:, idx_q] = np.squeeze(ev(nr, phi_q[idx_q]), -1)
    return W


def generate_noise(var_alpha, SNR, Q, P, rng=None):
    if rng is None:
        rng = np.random
    var_noise = var_alpha * 10**(-SNR/10)
    sigma = np.sqrt(var_noise / 2)
    return sigma * (rng.standard_normal((Q, P)) + 1j * rng.standard_normal((Q, P)))


class myarray(np.ndarray):
    @property
    def H(self):
        return self.conj().transpose()


def generate_channel_v2(nr, nt, angle_v, alpha_l):
    L = len(alpha_l)
    Hl = np.zeros((nr, nt, L), dtype=complex)
    for l in range(L):
        qq = ev(nt, angle_v[l]).view(myarray).H
        ww = ev(nr, angle_v[l+L])
        Hl[:, :, l] = alpha_l[l] * (ww * qq)
    Hl = np.sqrt(nt * nr) * Hl
    return np.sum(Hl, axis=-1)


def gaus2d(dist_m, dist_n, sigma):
    coeff = 1.0 / (2.0 * np.pi * sigma**2)
    exponent = -(dist_m**2 + dist_n**2) / (2.0 * sigma**2)
    return coeff * np.exp(exponent)


def get_real_imag(H):
    return np.dstack((np.real(H), np.imag(H)))


def generate_gt(L, amps, f1, f2, num_points_rx=256, num_points_tx=256, sigma=0.1, margin_factor=3.0):
    f1 = wrapTo2Pi(np.asarray(f1)); f2 = wrapTo2Pi(np.asarray(f2))
    margin = margin_factor * sigma
    p = np.linspace(-margin, 2*np.pi + margin, num_points_tx, endpoint=False)
    q = np.linspace(-margin, 2*np.pi + margin, num_points_rx, endpoint=False)
    Wp, Wq = np.meshgrid(p, q)
    mod = []
    for l in range(L):
        mod.append(amps[l] * gaus2d(Wp - f1[l], Wq - f2[l], sigma))
    return np.sum(mod, axis=0)


import scipy.ndimage

def data_generation(Training=True, sigma=0.07, M=256, condition=None, snr_low_bias=False, compute_gt=True):
    # snr_low_bias=True hole training-e beshi kore low-SNR sample dekhano hoy
    # (Part 1-e user-er 'low SNR-e aro valo koro' request-er jonno ekta lever)
    while True:
        if Training:
            L = np.random.randint(1, 10)
            if snr_low_bias:
                # triangular: beshi mass low-SNR-er dike, kintu pura range-i thake
                SNR = int(np.random.triangular(-15, -15, 25))
            else:
                SNR = np.random.randint(-15, 25)
            P = pyrandom.choice([16, 32])
            Q = P
            if P == 32:
                nt = pyrandom.choice([16, 32])
            else:
                nt = 16
            nr = nt
        else:
            if condition is None:
                raise ValueError("condition dorkar jokhon Training=False")
            L, SNR, QP, ntnr = condition
            P = Q = QP
            nt = nr = ntnr

        F = beamforming_vector_generation_P(P, nt)
        W = beamforming_vector_generation_Q(Q, nr)

        alpha_l = np.sqrt(1/L) * (np.random.randn(L) + 1j*np.random.randn(L)) / np.sqrt(2)
        alpha_l = alpha_l[np.argsort(-np.abs(alpha_l))]

        sep = np.pi/6
        points = generate_points(L, sep)
        phi_l = [pt[0] for pt in points]; psi_l = [pt[1] for pt in points]
        angle_v = np.hstack([phi_l, psi_l])

        omega_phi = np.pi*np.cos(phi_l); omega_psi = -np.pi*np.cos(psi_l)

        H = generate_channel_v2(nr, nt, angle_v, alpha_l)
        G = (W.view(myarray).H @ H) @ F
        Z = generate_noise(1.0, SNR, P, Q)
        Y = get_real_imag(G + Z)

        # eval (Training=False, compute_gt=False) somoy ei 256x256 Gaussian heatmap
        # kono kaje lagena (feat-i dorkar) -- tai onno somoy nosto kore banano hoy na
        if compute_gt:
            gt = generate_gt(L, np.ones(L), omega_phi, omega_psi, num_points_rx=M, num_points_tx=M, sigma=sigma)  # real amps -> no ComplexWarning
            gt = np.expand_dims(gt, -1).astype(np.float32)
        else:
            gt = None

        zoom = 4 if P == 16 else (2 if P == 32 else 1)
        data = np.dstack((scipy.ndimage.zoom(Y[:, :, 0], zoom, order=0),
                          scipy.ndimage.zoom(Y[:, :, 1], zoom, order=0))).astype(np.float32)

        if Training:
            yield np.real(data), np.real(gt)
        else:
            yield np.real(data), (np.real(gt) if gt is not None else None), np.stack([psi_l, phi_l]).astype(np.float32)

print('Part 1 physics + data generator ready.')

## Part 2 — Architecture: original ResNet vs SE-ResNet-Lite

Original `res_conv` block (base paper): Conv(5x5,12)->BN->ReLU->Conv(5x5,12)->BN
-> Add(skip) -> ReLU, **64-ta** eirokom block stack kora (469,393 param).

Notun `se_res_conv` block: same duita conv, kintu Add-er age ekta
**Squeeze-and-Excitation (SE) gate** jog kora -- global-average-pool kore
each channel-er "importance" shikhe, feature map-ke channel-wise scale kore.
Ei gate-tar নিজের param khub kom (12-channel-e ~87 extra param/block), tai
overhead prai negligible, kintu network-ke adaptively "kon channel-e
manage information ache" seta emphasize korar khomota dey -- bishesh kore
noisy (low-SNR) input-e eta relevant hote pare.

In [ ]:
def res_conv_original(x, filters=12):
    # base paper-er original block, verbatim
    x_skip = x
    x = Conv2D(filters, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, x_skip])
    x = Activation('relu')(x)
    return x


def Resnet_original(input_shape=(64,64,2), output_dim=1, n_blocks=64, filters=12):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(filters, 5, strides=2, padding='same')(x_in)
    for _ in range(n_blocks):
        x = res_conv_original(x, filters)
    x = Conv2DTranspose(output_dim, 5, strides=2, padding='same')(x)
    return Model(x_in, x, name=f'ResNet-{n_blocks}block')


def se_block(x, filters, ratio=4):
    se = GlobalAveragePooling2D()(x)
    se = Dense(max(filters // ratio, 1), activation='relu')(se)
    se = Dense(filters, activation='sigmoid')(se)
    se = Reshape((1, 1, filters))(se)
    return Multiply()([x, se])


def se_res_conv(x, filters=12, se_ratio=4):
    x_skip = x
    x = Conv2D(filters, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = se_block(x, filters, se_ratio)          # <-- notun jinis
    x = Add()([x, x_skip])
    x = Activation('relu')(x)
    return x


def SE_Resnet_Lite(input_shape=(64,64,2), output_dim=1, n_blocks=32, filters=12, se_ratio=4):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(filters, 5, strides=2, padding='same')(x_in)
    for _ in range(n_blocks):
        x = se_res_conv(x, filters, se_ratio)
    x = Conv2DTranspose(output_dim, 5, strides=2, padding='same')(x)
    return Model(x_in, x, name=f'SE-ResNet-Lite-{n_blocks}block')


print('Architecture functions ready.')

**Param count comparison** -- eikhane dekha jabe koto% kome ashe. `N_BLOCKS`
change kore (32, 24, 16 try kora jete pare) tumi nijer dorkar moto trade-off
বাছতে paro.

In [ ]:
RESNET_PARAMS_REF = 469_393     # base paper-er original ResNet (already verified)
N_BLOCKS = 32                     # <-- eikhane change korle param count change hobe

m_orig = Resnet_original(n_blocks=64)
m_se = SE_Resnet_Lite(n_blocks=N_BLOCKS)

p_orig = m_orig.count_params()
p_se = m_se.count_params()

print(f'Original ResNet (64 block, no attention):    {p_orig:,} param')
print(f'SE-ResNet-Lite  ({N_BLOCKS} block, +SE attention): {p_se:,} param')
print(f'Reduction: {(1 - p_se/p_orig)*100:.1f}% fewer parameters')
del m_orig, m_se   # memory bachate

## Part 3.0 -- Optional: warm-start from the pretrained original ResNet

`Resnet_original` (Part 2) ar tomar `SE_Resnet_Lite` (`N_BLOCKS=64`)-er
non-SE layer-gulo (Conv2D, BatchNormalization, ar duita end-er
Conv2DTranspose) shape-e hubohu same, ar duitai identical per-block
order-e create kora hoyeche -- tai egular pretrained weight
layer-by-layer manually copy kora jay, SE-gate (Dense) layer-gulo
bad diye (ogular kono pretrained equivalent nei, egula random-init-i
thakbe ar train-e shikhte hobe).

Ei cell chalanor age lagbe: `models/inf_model_007_256_resnet.h5`
file-tar path -- ei-ta DL_DOA_CLONE repo/dataset attach kora thakle pabe
(age-r ResNet-reproduction notebook-e jei path use hoyeche shei-i).

In [ ]:
def transfer_from_pretrained_resnet(se_model, pretrained_path, n_blocks=64):
    # Original architecture-ta ekhane-i rebuild kore, real pretrained weight
    # load kora hocche -- eta already-verified ResNet reproduction-er shathe
    # identical (same Resnet_original function).
    orig = Resnet_original(n_blocks=n_blocks)
    orig.load_weights(pretrained_path)

    # Duita model-eri weighted layer-gulo (Conv2D/BatchNorm/Conv2DTranspose)
    # creation-order-e ekdom match kore -- Dense (SE-gate-er notun layer)
    # bad diye rest shobgulo positionally align kora jay.
    orig_weighted = [l for l in orig.layers if l.get_weights()]
    se_weighted = [l for l in se_model.layers
                   if l.get_weights() and not isinstance(l, tf.keras.layers.Dense)]

    assert len(orig_weighted) == len(se_weighted), (
        f'layer count mismatch: original={len(orig_weighted)} vs se-model={len(se_weighted)} '
        f'-- N_BLOCKS eki (64) ache to duita model-e?')

    n_copied = 0
    for lo, ls in zip(orig_weighted, se_weighted):
        ls.set_weights(lo.get_weights())
        n_copied += 1

    n_se_dense = sum(1 for l in se_model.layers if isinstance(l, tf.keras.layers.Dense))
    print(f'Transferred {n_copied} matching layer(s) from pretrained ResNet '
          f'(Conv2D/BatchNorm/Conv2DTranspose).')
    print(f'{n_se_dense} SE-gate Dense layer(s) remain randomly initialized '
          f'-- these + light fine-tuning still need training.')
    del orig
    return se_model


# --- Usage (uncomment ar path thik kore run koro) ---
# N_BLOCKS = 64; SE_RATIO = 4
# model = SE_Resnet_Lite(n_blocks=N_BLOCKS, se_ratio=SE_RATIO)
# model = transfer_from_pretrained_resnet(model, 'DL_DOA/models/inf_model_007_256_resnet.h5', n_blocks=N_BLOCKS)
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='mse')
# # ekhon Part 3-er training loop cholao, kintu EPOCHS onek kom rakho (jemon 8-15)
# # -- base feature-extraction already pretrained, shudhu SE-gate + light adaptation shikhte hobe

## Part 3 — Training

Base paper-er nijer training generator (`data_generation`) hubohu copy kore
use kora hocche -- random `L` (1-9), random `SNR` (-15..25 dB), random
`P/Q/Nt/Nr` -- jate training distribution base paper-er ResNet-er shathe
identical thake (fair comparison-er jonno guruttopurno).

`SNR_LOW_BIAS = True` korle training-e beshi kore low-SNR sample dekhano hobe
(Part 1-er `snr_low_bias` flag) -- ekta shohoj lever try korar jonno, jodi
lokkho hoy specifically low-SNR-e improvement dekhano.

**Guruttopurno:** ei cell-er default STEPS_PER_EPOCH/EPOCHS just ekta SMOKE
TEST-er jonno choto rakha ache (dekhar jonno shob thik chole kina). REAL
result-er jonno eগুলো onek baড়িয়ে (jemon original-er kachakachi scale-e)
tomar Kaggle GPU-te chalate hobe.

In [ ]:
N_BLOCKS = 32          # koyta residual block (64 = original size, kom = lighter)
SE_RATIO = 4            # SE bottleneck reduction ratio
SNR_LOW_BIAS = False    # True korle training-e beshi low-SNR sample dekhano hobe

BATCH = 8
STEPS_PER_EPOCH = 20    # SMOKE TEST value -- real run-e 500-2000+ korte hobe
EPOCHS = 2              # SMOKE TEST value -- real run-e 30-100+ korte hobe

model = SE_Resnet_Lite(n_blocks=N_BLOCKS, se_ratio=SE_RATIO)
print(model.name, '-', model.count_params(), 'trainable param')

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

def make_tf_dataset(batch_size, snr_low_bias=False):
    gen = data_generation(Training=True, snr_low_bias=snr_low_bias)
    def gen_fn():
        for data, gt in gen:
            yield data, gt
    ds = tf.data.Dataset.from_generator(
        gen_fn,
        output_signature=(tf.TensorSpec(shape=(64, 64, 2), dtype=tf.float32),
                          tf.TensorSpec(shape=(256, 256, 1), dtype=tf.float32)))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(BATCH, snr_low_bias=SNR_LOW_BIAS)

t0 = time.time()
history = model.fit(train_ds, steps_per_epoch=STEPS_PER_EPOCH, epochs=EPOCHS, verbose=1)
print(f'\ntraining done in {time.time()-t0:.0f}s -- {STEPS_PER_EPOCH*EPOCHS*BATCH:,} samples seen')
print('loss history:', [f'{l:.5f}' for l in history.history['loss']])
model.save_weights('se_resnet_lite.weights.h5')

## Part 3.1 — Continue training from saved weights (optional)

Notun kore shuru na kore, age-r `se_resnet_lite.weights.h5` theke continue
kora -- shomoy bachanor jonno. **Guruttopurno:** architecture config
(`N_BLOCKS`, `SE_RATIO`, `filters`) Part 3-er shathe HUBOHU MATCH korte hobe,
naile `load_weights` shape-mismatch error dibe. Jodi kernel restart hoye
thake, age Part 0-2 (setup, physics, architecture function-gulo) re-run kore
niyo, tarpor ei cell.

In [ ]:
MORE_EPOCHS = 20          # koto beshi epoch chalate chao
NEW_LR = 5e-4              # fine-tuning-er jonno LR ektu kom rakha bhalo

model2 = SE_Resnet_Lite(n_blocks=N_BLOCKS, se_ratio=SE_RATIO)   # Part 3-er shathe match hote hobe
model2.load_weights('se_resnet_lite.weights.h5')
model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=NEW_LR), loss='mse')

train_ds2 = make_tf_dataset(BATCH, snr_low_bias=SNR_LOW_BIAS)

t0 = time.time()
history2 = model2.fit(train_ds2, steps_per_epoch=STEPS_PER_EPOCH, epochs=MORE_EPOCHS, verbose=1)
print(f'\ncontinued training done in {time.time()-t0:.0f}s -- {MORE_EPOCHS*STEPS_PER_EPOCH*BATCH:,} more samples seen')
print('loss history (continued):', [f'{l:.5f}' for l in history2.history['loss']])

model2.save_weights('se_resnet_lite_v2.weights.h5')   # notun file -- purono checkpoint-o thakbe
model = model2   # Part 4 evaluation ei 'model' variable-i use kore -- update kore dilam, tai
                 # niche shudhu Part 4 (evaluation) cell-gulo abar run korle notun model-i test hobe
print('\n`model` variable update kora holo -- ekhon Part 4 (evaluation) abar run koro.')

## Part 4 — Evaluation (same protocol, same metric as base paper's ResNet)

`run_inference_and_metrics_resnet()`-er hubohu shei protocol: L=3, SNR -10..25,
P=Q=Nt=Nr=16, sigma=0.07, M=256, top-L blob by amplitude, RMSE-within-1-degree
+ Pd. Ei function-tao repo theke na niye nijer moto banano hoyeche (self-contained
rakhar jonno), kintu logic hubohu shei.

In [ ]:
def get_blob_detector():
    params = cv2.SimpleBlobDetector_Params()
    params.filterByColor = True; params.blobColor = 255
    params.minThreshold = 0; params.maxThreshold = 255
    params.filterByArea = True; params.minArea = 1; params.maxArea = 1000
    params.filterByCircularity = False; params.filterByConvexity = False; params.filterByInertia = False
    return cv2.SimpleBlobDetector_create(params)


def get_blob_peaks(pred_2d, detector):
    img_norm = cv2.normalize(pred_2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    keypoints = detector.detect(img_norm)
    coords = np.array([kp.pt for kp in keypoints])
    if len(coords) == 0:
        return np.zeros((0, 2)), np.zeros((0,))
    coords_r = np.round(coords).astype(int)
    amps = np.array([img_norm[y, x] if 0 <= y < img_norm.shape[0] and 0 <= x < img_norm.shape[1] else 0
                     for (x, y) in coords_r])
    order = np.argsort(-amps)
    return coords[order], amps[order]


def wrap_2pi_to_minus_pi(a):
    a = np.asarray(a)
    return np.where(a > np.pi, a - 2*np.pi, a)


def peaks_to_angles(peaks, sigma=0.07, grid_size=256, margin_factor=3.0):
    if peaks.shape[0] == 0:
        return np.array([]), np.array([])
    margin = margin_factor * sigma
    ext = 2*np.pi + 2*margin
    freqs = wrap_2pi_to_minus_pi(-margin + (peaks.T / grid_size) * ext)
    psi_est = np.arccos(np.clip(-freqs[1]/np.pi, -1, 1))
    phi_est = np.arccos(np.clip(freqs[0]/np.pi, -1, 1))
    return psi_est, phi_est


def permute_pairs(A, B):
    from scipy.optimize import linear_sum_assignment
    A = np.asarray(A); B = np.asarray(B)
    dist = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
    r, c = linear_sum_assignment(dist)
    return [(tuple(A[i]), tuple(B[j])) for i, j in zip(r, c)]


def prepare_for_metric(angles_est, feat):
    L = feat.shape[-1]
    if len(angles_est[0]) < L:
        return np.array([feat[0], feat[1]]), np.array([np.full(L, np.nan), np.full(L, np.nan)])
    est = (angles_est[0][:L], angles_est[1][:L])
    pairs_est = list(zip(est[0], est[1])); pairs_true = list(zip(feat[0], feat[1]))
    permuted = permute_pairs(pairs_true, pairs_est)
    first = [p[0] for p in permuted]; second = [p[1] for p in permuted]
    psi_t, phi_t = zip(*first); psi_e, phi_e = zip(*second)
    return np.array([psi_t, phi_t]), np.array([psi_e, phi_e])


def get_ang_difference(gt, pred):
    return (np.angle(np.exp(1j*gt) * np.exp(-1j*pred)) * 180/np.pi).flatten()


def filter_angles(diff, max_deg=1.0):
    return diff[np.abs(diff) <= max_deg], diff[np.abs(diff) > max_deg]


print('Eval utilities ready.')

In [ ]:
SNRS_EVAL = list(range(-10, 30, 5))
N_PER_SNR_EVAL = 30     # SMOKE TEST value -- real run-e 1000 (base paper-er moto) korte hobe

detector = get_blob_detector()

def evaluate_model(model, n_per_snr=N_PER_SNR_EVAL, snrs=SNRS_EVAL, L=3, P=16, ntnr=16, sigma=0.07, M=256):
    rmse, pd_ = {}, {}
    for snr in snrs:
        gen = data_generation(Training=False, sigma=sigma, M=M, condition=(L, snr, P, ntnr), compute_gt=False)
        good, bad = [], []
        for _ in range(n_per_snr):
            data, gt, feat = next(gen)
            pred = model(tf.expand_dims(data, 0), training=False)[0, :, :, 0].numpy()
            peaks, amps = get_blob_peaks(pred, detector)
            peaks = peaks[:L]
            angles_est = peaks_to_angles(peaks, sigma=sigma, grid_size=M)
            gt_a, pr_a = prepare_for_metric(angles_est, feat)
            if np.isnan(pr_a).any():
                bad.append(np.full(gt_a.size, 999.0)); continue
            g, b = filter_angles(get_ang_difference(gt_a, pr_a), 1.0)
            good.append(g); bad.append(b)
        good = np.concatenate(good) if good else np.array([])
        bad = np.concatenate(bad) if bad else np.array([])
        tot = len(good) + len(bad)
        rmse[snr] = np.sqrt(np.mean(good**2)) if len(good) else np.nan
        pd_[snr] = len(good)/tot if tot else np.nan
    return rmse, pd_

t0 = time.time()
se_rmse, se_pd = evaluate_model(model)
print(f'evaluated in {time.time()-t0:.0f}s\n')
for s in SNRS_EVAL:
    print(f'SNR={s:4d}  SE-ResNet-Lite RMSE={se_rmse[s]:.4f}  Pd={se_pd[s]:.4f}')

## Part 5 — Comparison against the verified original ResNet reference

Same L=3, P=16, Ntnr=16 condition, same real-weight-verified reference table
used throughout this project.

In [ ]:
RESNET_REF_RMSE = {-10:0.5532,-5:0.5118,0:0.4581,5:0.3920,10:0.3256,15:0.2792,20:0.2528,25:0.2377}
RESNET_REF_PD   = {-10:0.2043,-5:0.4366,0:0.6396,5:0.7790,10:0.8623,15:0.8987,20:0.9250,25:0.9378}

print('='*78)
print(f'{"SNR":>5} | {"SE-Lite RMSE":>12} {"SE-Lite Pd":>10} | {"ResNet RMSE":>11} {"ResNet Pd":>9} | {"d-Pd":>7}')
print('-'*78)
for s in SNRS_EVAL:
    dpd = se_pd[s] - RESNET_REF_PD[s]
    print(f'{s:>5} | {se_rmse[s]:>12.4f} {se_pd[s]:>10.4f} | {RESNET_REF_RMSE[s]:>11.4f} {RESNET_REF_PD[s]:>9.4f} | {dpd:>+7.4f}')
print('='*78)
print(f'\nParam count: SE-ResNet-Lite = {model.count_params():,}   vs   original ResNet = {RESNET_PARAMS_REF:,}'
      f'  ({(1-model.count_params()/RESNET_PARAMS_REF)*100:.1f}% fewer)')

low_snr = [-10, -5]
mean_low_delta = np.nanmean([se_pd[s]-RESNET_REF_PD[s] for s in low_snr])
print(f'Mean Pd delta at low SNR ({low_snr}): {mean_low_delta:+.4f}  (>0 mane SE-ResNet-Lite low-SNR-e ভালো)')

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
axs[0].plot(SNRS_EVAL, [se_rmse[s] for s in SNRS_EVAL], '^-', color='crimson', label='SE-ResNet-Lite (this run)')
axs[0].plot(SNRS_EVAL, [RESNET_REF_RMSE[s] for s in SNRS_EVAL], 's-', label='ResNet (reference)')
axs[0].axhline(1/np.sqrt(3), ls='--', c='gray', label='chance floor')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)'); axs[0].set_title('RMSE vs SNR'); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].plot(SNRS_EVAL, [se_pd[s] for s in SNRS_EVAL], '^-', color='crimson', label='SE-ResNet-Lite (this run)')
axs[1].plot(SNRS_EVAL, [RESNET_REF_PD[s] for s in SNRS_EVAL], 's-', label='ResNet (reference)')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd'); axs[1].set_ylim(-.02, 1.02)
axs[1].set_title('Detection probability vs SNR'); axs[1].legend(); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Part 6 — Honest summary

Ei run-er number gulo **shudhu ekta smoke-test scale-e** (khub kom step,
khub kom eval sample) -- final verdict deওয়ার moto na. Real result janar
jonno:

1. `N_BLOCKS` (32/24/16), `SE_RATIO`, `SNR_LOW_BIAS` -- ei config-gulo dorkar
   moto adjust koro.
2. Part 3-er `STEPS_PER_EPOCH`/`EPOCHS` onek baড়াও (original ResNet-er
   training scale-er kachakachi -- exact na jana thakleo, jotoটা shomvob
   beshi samples dekhano bhalo).
3. Part 4-er `N_PER_SNR_EVAL` = 1000 koro (base paper-er nijer eval scale)
   jate reported number-ta statistically shothik hoy.
4. Full run-ta tomar Kaggle GPU-te chalao, tarpor ei Part 5-er table/plot-ta
   dekhe decide koro: (a) param komeo accuracy ধরে rakhte parlo kina,
   (b) attention jog kore even minor accuracy gain elo kina, (c) low-SNR-e
   বিশেষ কোনো সুবিধা holo kina (`mean_low_delta` positive hole ha)।